# 객체지향 문법: 다형성 (Polymorphism) 

## 🎯 학습 목표

- **다형성**이 무엇인지 "하나의 명령, 여러 결과"로 설명할 수 있다.
- `if / elif` 로 분기하던 코드를 **다형성으로 리팩터링**할 수 있다.
- 여러 종류의 객체를 **하나의 리스트에 담아 for 문 하나로** 처리할 수 있다.
- `NotImplementedError` 와 `abc` 모듈로 **자식에게 구현을 강제**할 수 있다.
- **덕 타이핑(duck typing)** 이 무엇인지 설명할 수 있다.
- `__str__` 로 객체의 출력 형태를 바꿀 수 있다.


> 🔗 **이전 학습과의 연결**:  `ApiClient` → `OpenAIClient`, `GeminiClient` 상속 구조를 만들었습니다.
> 오늘은 그 자식들을 **하나의 리스트에 담아 똑같이 다루는 법**을 배웁니다. 상속은 **준비**, 다형성은 **활용**입니다.

---

## 0. 왜 다형성이 필요한가요?

### 😱 문제 상황: if 문 지옥 (if-hell)

AI 모델을 3개 연동한 코드입니다. 새 모델을 추가하면 무슨 일이 벌어질까요?

In [ ]:
# ✍️ 아래 코드를 실행하고, 새 모델 'llama' 를 추가하려면 무엇을 고쳐야 하는지 적어 보세요.
def ask_ai(service, prompt):
    if service == "openai":
        return f"[OpenAI] {prompt}"
    elif service == "gemini":
        return f"[Gemini] {prompt} (안전필터 통과)"
    elif service == "claude":
        return f"[Claude] {prompt}"
    else:
        return "지원하지 않는 서비스입니다"

for s in ["openai", "gemini", "claude", "llama"]:
    print(ask_ai(s, "파이썬이 뭐야?"))

# 📌 새 모델을 추가하려면 어디를 고쳐야 하나요? elif문을 작성해서 새로운 모델을 넣는다.
# 📌 모델이 20개가 되면 이 함수는 몇 줄이 될까요? 모델 하나당 2중 20개면 40줄 +else 2줄

### 🎭 다형성이란?

> **다형성(Polymorphism)** = "多(많은) 形(모양) 性(성질)"
> **하나의 이름(명령)** 이 **객체에 따라 다른 동작**을 하는 것

### 🔌 비유 ① 전원 버튼

- TV, 선풍기, 노트북의 전원 버튼은 모두 **"켠다"** 는 같은 명령입니다.
- 그런데 결과는 제각각입니다. TV는 화면이 켜지고, 선풍기는 날개가 돌죠.
- 우리는 **기기마다 다른 사용법을 외우지 않아도** 됩니다. "전원 버튼을 누른다"만 알면 되죠.

### 🐾 비유 ② "말해봐!"

- 개·고양이·소에게 **"말해봐"** 라고 똑같이 말합니다.
- 멍멍, 야옹, 음메 — **결과는 다르지만 명령은 하나**입니다.

### 🔑 핵심 아이디어

| if 문 방식 | 다형성 방식 |
|---|---|
| **부르는 쪽**이 종류를 다 알아야 함 | 부르는 쪽은 **메서드 이름만** 알면 됨 |
| 새 종류 추가 → **기존 코드 수정** | 새 종류 추가 → **클래스만 추가** |
| if 가 계속 늘어남 | 코드 길이 그대로 |

> 💡 실무 원칙: **OCP (Open-Closed Principle)** — "확장에는 열려 있고, 수정에는 닫혀 있어야 한다"

---

## 1. 다형성의 핵심 코드: 리스트 + for 문

### 📖 원리

1. 부모 클래스에 **공통 메서드 이름**을 정합니다. (예: `speak()`)
2. 자식마다 그 메서드를 **오버라이딩**합니다. 
3. 여러 자식 객체를 **한 리스트에 담고 for 문 하나로** 호출합니다.

In [6]:
# ✍️ 빈칸을 채우세요
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        print(f"{self.name}: ...")


class Dog(Animal):
    def speak(self):                       # 빈칸: 부모와 같은 이름의 메서드
        print(f"{self.name}: 멍멍!")


class Cat(Animal):
    def speak(self):
        print(f"{self.name}: 야옹~")


class Cow(Animal):
    def speak(self):
        print(f"{self.name}: 음메~")


animals = [Dog("초코"), Cat("나비"), Cow("얼룩이")]   # 빈칸: 세 번째 클래스

for a in animals:
    a.speak()                              # 빈칸: 공통 메서드 이름

초코: 멍멍!
나비: 야옹~
얼룩이: 음메~


### 🔍 왜 if 가 필요 없을까요?

파이썬은 `a.speak()` 를 실행할 때 **a 가 실제로 어떤 클래스의 객체인지 보고** 알맞은 메서드를 찾아갑니다.
즉 **판단을 파이썬에게 맡기는 것**입니다. 개발자가 if 로 일일이 분기할 필요가 없습니다.

### ✨ 새 종류를 추가해 봅시다

`Duck` 클래스를 추가해도 **for 문은 한 글자도 고치지 않습니다.**

In [7]:
# ✍️ 직접 해 보세요
# 1) Duck(Animal) 클래스를 만들고 speak() 오버라이딩 → "이름: 꽥꽥!"
class Duck(Animal):
    def speak(self):
        print(f"{self.name}: 꽥꽥!")
# 2) animals 리스트에 Duck 객체를 append
animals.append(Duck('오리'))
# 3) 기존 for 문을 "수정하지 않고" 다시 실행
for a in animals:
    a.speak()
# 📌 for 문을 고쳐야 했나요?

초코: 멍멍!
나비: 야옹~
얼룩이: 음메~
오리: 꽥꽥!


---

## 2. 자식에게 구현을 강제하기

### 😰 문제: 자식이 오버라이딩을 깜빡하면?

`speak()` 를 구현하지 않은 자식이 끼어들면, 부모의 기본 동작이 **조용히** 실행됩니다.
버그를 늦게 발견하게 되죠.

In [8]:
# ✍️ Fish(Animal) 클래스를 speak() 없이 만들고 speak() 를 호출해 보세요.
class Fish(Animal):
    pass
animals.append(Fish('물코기'))
for a in animals:
    a.speak()
# 📌 어떤 일이 벌어졌나요? 이게 왜 위험할까요? 부모의 speak()를 실행한다. => 부모의 기본동작이 구현된다.

초코: 멍멍!
나비: 야옹~
얼룩이: 음메~
오리: 꽥꽥!
물코기: ...


### 🔑 해결책 1: `NotImplementedError` 로 알려 주기

부모 메서드에서 **일부러 에러를 발생**시켜 "자식이 꼭 만들어야 한다"고 알립니다.

In [10]:
# ✍️ 다이어그램을 분석하여 클래스를 완성하세요
class Animal:
    def __init__(self, name):
        self.name =name

    def speak(self):
        raise NotImplementedError(f"{type(self).__name__}에는 speak() 정의가 필요합니다.")

class Dog(Animal):
    pass
class Fish(Animal):
    pass
f =Fish('물코기')
f.speak()


# Dog("초코").speak()

# try:
#     Fish("니모").speak()
# except NotImplementedError as e:                   # 빈칸: 위에서 발생시킨 에러 종류
#     print("❌ 에러:", e)

NotImplementedError: Fish에는 speak() 정의가 필요합니다.

### 🔑 해결책 2: `abc` 모듈 — 더 강력한 강제 (추상 클래스)

- `ABC` 를 상속하고 `@abstractmethod` 를 붙이면, **객체를 만드는 순간부터** 막아 줍니다.
- `NotImplementedError` 는 **메서드를 호출할 때** 에러가 나지만, 추상 클래스는 **객체 생성 시점**에 막습니다. 더 빠르게 잡아 주죠.

### 📐 비유: 설계도 양식

- "면적 계산 칸은 **반드시** 채워서 제출하세요"라고 적힌 양식입니다.
- 빈칸으로 제출하면 **접수 단계에서 반려**됩니다.

In [15]:
# ✍️ 다이어그램을 분석하여 클래스를 완성하세요
from abc import ABC, abstractmethod
# 도형 클래스(Shape)를 추상클래스로 설계해 
class Shape(ABC):
    def __init__(self,name):
        self.name =name

# 1. 면적을 구하는 기능(area()), 구한 면적을 출력하는 기능(describe())를 구현
    @abstractmethod
    def area(self):
        pass

    def describe(self):
        print(f"{self.name}의 면적은, {self.area():.2f}")
# 2. Shape 클래스를 상속한 Square(정사각형) 클래스를 구현하세요.
class Square(Shape):
    def __init__(self,side):
        super().__init__('정사각형')
        self.side =side
    def area(self):
        return self.side**2




Square(5).describe()                   # 기대: 정사각형의 면적: 25.00

class Broken(Shape):
    pass

try:
    Broken("미완성")                    # 어떤 에러가 날까요?
except TypeError as e:
    print("❌ 객체 생성 차단:", e)

정사각형의 면적은, 25.00
❌ 객체 생성 차단: Can't instantiate abstract class Broken with abstract method area


---

## 3. 덕 타이핑(Duck Typing): 상속이 없어도 다형성이 된다

### 🦆 유명한 격언

> **"오리처럼 걷고 오리처럼 꽥꽥거리면, 그것은 오리다."**
> (If it walks like a duck and quacks like a duck, it's a duck.)

### 📖 개념

- 파이썬은 **"무슨 클래스냐"** 를 따지지 않습니다. **"그 메서드가 있냐"** 만 봅니다.
- 그래서 **부모가 달라도, 상속 관계가 전혀 없어도** 같은 메서드 이름만 있으면 다형성이 됩니다.
- 이것이 파이썬 다형성의 가장 큰 특징입니다. (Java 등과 다른 점)

In [18]:
# ✍️ 직접 작성해 보세요
# 1) 상속 관계가 전혀 없는 Duck, Robot, Piano 클래스를 만들고
#    각각 sound() 메서드만 똑같은 이름으로 만들기 (출력 내용은 자유)
class Duck:
    def sound(self):
        return print("꽥꽥")
class Robot:
    def sound(self):
        return print("로봇로봇")
class Piano:
    def sound(self):
        return print("피아피아")
    
# 2) 세 객체를 리스트에 담아 for 문 하나로 sound() 호출
for thing in [Duck(),Robot(),Piano()]:
    thing.sound()

# 📌 공통 부모가 없는데도 동작하는 이유는? 메서드 이름이 

꽥꽥
로봇로봇
피아피아


### ⚠️ 덕 타이핑의 주의점

메서드가 없는 객체가 섞이면 **실행 중에** 에러가 납니다. 미리 확인하려면 `hasattr()` 을 씁니다.

In [21]:
# ✍️ 빈칸을 채우세요
class Rock:
    pass


things = [Duck(), Rock(), Piano()]

for t in things:
    if hasattr(t, "sound"):                 # 빈칸: 속성/메서드 존재 확인 함수
        t.sound()
    else:
        print(f"{type(t).__name__}: 소리를 낼 수 없어요")

꽥꽥
Rock: 소리를 낼 수 없어요
피아피아


---

## 4. `__str__`: 객체를 print 할 때의 모습 바꾸기 

### 📖 개념

- 객체를 그냥 `print()` 하면 `<__main__.Dog object at 0x7f...>` 처럼 알아보기 힘든 글자가 나옵니다.
- `__str__` 메서드를 정의하면 **print 할 때 보일 문장**을 직접 정할 수 있습니다.
- `print()` 라는 **하나의 명령**이 클래스마다 **다른 결과**를 내므로, 이것도 다형성입니다.

> 💡 `__str__` 은 다음 차시 **연산자 중복(operator overloading)** 의 입문입니다.

In [24]:
# ✍️ 빈칸을 채우세요
class Dog:
    def __init__(self, name):
        self.name = name


class Cat:
    def __init__(self, name):
        self.name = name

    def __str__(self):                          # 빈칸: print 할 때 호출되는 특별 메서드
        return f"🐱 귀여운 고양이 {self.name}"


print(Dog("초코"))
print(Cat("나비"))
# 📌 두 출력의 차이는? __str__ 을 설정하지 않으면 객체와 객체에 대한 값이 16진수로 나오고 
# __str__ 로 내용을 overloading? overriding? 해두면 설정한 반환값이 나온다.

🐱 귀여운 고양이 나비


---

## 5. 🛠️ 단계별 실습 과제

### 실습 1 [초급~중급] 도형 면적 계산기 📐

> ** 정삼각형 `Figure` 클래스가 여기서 완성됩니다!**

**요구사항**

- 추상 클래스 `Shape(ABC)`: `__init__(self, name)`, 추상 메서드 `area()`
- `EquilateralTriangle(Shape)`: 한 변 `side`, 면적 = **(√3 / 4) × side²**
- `Rectangle(Shape)`: `width`, `height`, 면적 = width × height
- `Circle(Shape)`: `radius`, 면적 = π × radius²
- 세 도형을 **한 리스트에 담아 for 문 하나로** 면적 출력 (소수점 2자리)
- 전체 면적의 **합계**도 출력

In [25]:
# ✍️ 실습 1: 빈칸을 채워 완성하세요
import math
from abc import ABC, abstractmethod


class Shape(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def area(self):
        pass


class EquilateralTriangle(Shape):
    def __init__(self, side):
        super().__init__("정삼각형")
        self.side = side

    def area(self):
        return (math.sqrt(3) / 4) * self.side ** 2    # 빈칸: 정삼각형 공식의 분모


class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__("직사각형")
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height                                       # 빈칸: 직사각형 면적


class Circle(Shape):
    def __init__(self, radius):
        super().__init__("원")
        self.radius = radius

    def area(self):
        return math.pi*self.radius**2                                       # 빈칸: 원의 면적 (math.pi 사용)


shapes = [EquilateralTriangle(10), Rectangle(10, 5), Circle(3)]

for s in shapes:
    print(f"{s.name}: {s.area():.2f}")                    # 빈칸: 공통 메서드 이름

total = sum(s.area() for s in shapes)
print(f"전체 면적 합계: {total:.2f}")
# 기대값: 정삼각형 43.30 / 직사각형 50.00 / 원 28.27

정삼각형: 43.30
직사각형: 50.00
원: 28.27
전체 면적 합계: 121.58


### 실습 2 [중급] if 지옥 탈출! AI 라우터 리팩터링 🔀

> **0절의 `ask_ai()` 함수를 다형성으로 다시 만듭니다.** 4차시 상속 구조를 그대로 활용합니다.

**요구사항**

- 추상 부모 `BaseClient(ABC)`
  - `__init__(self, service_name, api_key)`: `service_name`, `__api_key`(private), `_call_count = 0`
  - 추상 메서드 `ask(self, prompt)`
  - `get_masked_key()`: 앞 4글자 + 나머지 `*`
  - `get_call_count()`: 호출 횟수 리턴
- 자식 3개: `OpenAIClient`, `GeminiClient`, `ClaudeClient`
  - 각각 서비스명을 고정하고 `ask()` 를 자기 방식으로 구현 (`_call_count` 증가 포함)
- **라우터 함수** `ask_all(clients, prompt)`
  - `if` 를 **한 번도 쓰지 않고** 모든 클라이언트에게 같은 질문을 던져 결과를 출력
- 새 모델 `LlamaClient` 를 추가했을 때 **`ask_all` 을 고치지 않아도** 되는지 확인

In [ ]:
# ✍️ 실습 2: 요구사항을 보고 완성하세요
from abc import ABC, abstractmethod


class BaseClient(ABC):
    def __init__(self, service_name, api_key):
        # TODO 1: service_name, __api_key(private), _call_count=0
        self.service_name =service_name
        self.__api_key =api_key
        self._call_count =0

    @abstractmethod
    def ask(self, prompt):
        pass   # 추상 메서드 (내용 없음)

    def get_masked_key(self):
        # TODO 2: 앞 4글자 + 나머지 '*'
        return self.__api_key[:4]+('*' * len(self.__api_key)-4)

    def get_call_count(self):
        # TODO 3: 호출 횟수 return
        return self._call_count


class OpenAIClient(BaseClient):
    def __init__(self, api_key, model="gpt-4o"):
        # TODO 4: super() 로 "OpenAI" 고정 + model 저장
        super().__init__('OpenAI',api_key)
        self.model =model

    def ask(self, prompt):
        # TODO 5: 호출수 증가 + "[OpenAI/모델명] 프롬프트" return
        self._call_count+=1
        return f"[{self.service_name}/{self.model}] {prompt}"


class GeminiClient(BaseClient):
    def __init__(self, api_key):
        # TODO 6
        super().__init__('Gemini',api_key)

    def ask(self, prompt):
        # TODO 7: "[Gemini] 프롬프트 (안전필터 통과)"
        self._call_count+=1
        return f"[{self.service_name}] {prompt} (안전필터 통과)"


class ClaudeClient(BaseClient):
    def __init__(self, api_key):
        # TODO 8
        super().__init__('Claude',api_key)

    def ask(self, prompt):
        # TODO 9: "[Claude] 프롬프트 (근거와 함께 답변)"
        self._call_count+=1
        return f"[{self.service_name}] {prompt} (근거와 함께 답변)"


def ask_all(clients, prompt):
    # TODO 10: if 를 쓰지 않고 모든 클라이언트의 ask() 결과를 출력
    for item in clients:
        print(item.ask(prompt))


# 테스트 코드 (수정 금지)
clients = [
    OpenAIClient("sk-aaaa1111"),
    GeminiClient("gm-bbbb2222"),
    ClaudeClient("cl-cccc3333"),
]
ask_all(clients, "파이썬이 뭐야?")

print()
# 🔥 도전: LlamaClient 를 추가해도 ask_all 을 고치지 않아도 되는지 확인해 보세요
class LlamaClient(BaseClient):
    def __init__(self, api_key):
        super().__init__('Llama', api_key)
    def ask(self, prompt):
        self._call_count +=1
        return f"[{self.service_name}] {prompt} (번역으로 답변)"
    
clients.append(LlamaClient('ll-dddd4444'))
ask_all(clients,"라마가 뭐야?")

[OpenAI/gpt-4o] 파이썬이 뭐야?
[Gemini] 파이썬이 뭐야? (안전필터 통과)
[Claude] 파이썬이 뭐야? (근거와 함께 답변)

[OpenAI/gpt-4o] 라마가 뭐야?
[Gemini] 라마가 뭐야? (안전필터 통과)
[Claude] 라마가 뭐야? (근거와 함께 답변)
[Llama] 라마가 뭐야? (번역으로 답변)


### 실습 3 [중급] 통합 챗봇 매니저 🤖

> 4차시의 `TranslateBot`, `SummaryBot` 을 **매니저 클래스로 통합 관리**합니다.
> 실제 AI 서비스의 **멀티 에이전트 구조**와 같은 설계입니다.

**요구사항**

- 추상 부모 `BaseBot(ABC)`
  - `__init__(self, name)`: `name`, `_log = []`(protected)
  - 추상 메서드 `handle(self, message)` → 처리 결과 문자열 리턴
  - `run(self, message)`: `_log` 에 기록하고 `"[이름] handle 결과"` 를 **출력**
  - `log_count()`: 기록 개수 리턴
- 자식 3개
  - `EchoBot`: `"그대로: 메시지"`
  - `UpperBot`: `"대문자: 메시지를 대문자로"` (힌트: `.upper()`)
  - `LengthBot`: `"글자수: N자"`
- `BotManager` 클래스 (상속 아님 — **포함(composition)** 관계!)
  - `__init__`: 빈 리스트 `self.bots`
  - `add(self, bot)`: 봇 등록
  - `broadcast(self, message)`: 등록된 **모든 봇에게** 같은 메시지를 전달 (if 없이!)
  - `report(self)`: 각 봇의 이름과 기록 개수 출력

In [35]:
# ✍️ 실습 3: 요구사항을 보고 완성하세요
from abc import ABC, abstractmethod


class BaseBot(ABC):
    def __init__(self, name):
        # TODO 1: name, _log=[] (protected)
        self.name =name
        self._log=[]

    @abstractmethod
    def handle(self, message):
        pass

    def run(self, message):
        # TODO 2: _log 에 기록 → "[이름] {self.handle(message)}" 출력
        self._log.append(message)
        return print(f"[{self.name}] {self.handle(message)}")

    def log_count(self):
        # TODO 3: 기록 개수 return
        return len(self._log)


class EchoBot(BaseBot):
    # TODO 4: handle() → "그대로: 메시지"
    def handle(self, message):
        return f"그대로: {message}"


class UpperBot(BaseBot):
    # TODO 5: handle() → "대문자: 메시지를 대문자로"  (힌트: .upper())
    def handle(self,message):
        return f"대문자: {message.upper()}"

class LengthBot(BaseBot):
    # TODO 6: handle() → "글자수: N자"  (힌트: len())
    def handle(self, message):
        return f"글자수: {len(message)}"


class BotManager:
    def __init__(self):
        # TODO 7: self.bots = []
        self.bots =[]

    def add(self, bot):
        # TODO 8: 리스트에 추가 + "✅ 이름 등록 완료" 출력
        self.bots.append(bot)
        print(f'"✅ 이름 등록 완료"')

    def broadcast(self, message):
        # TODO 9: 모든 봇의 run(message) 호출 (if 사용 금지!)
        for m in self.bots:
            m.run(message)

    def report(self):
        # TODO 10: 각 봇의 이름과 기록 개수 출력
        for item in self.bots:
            print(f"[{item.name}] 기록 개수:{item.log_count()}")


# 테스트 코드 (수정 금지)
manager = BotManager()
manager.add(EchoBot("에코봇"))
manager.add(UpperBot("대문자봇"))
manager.add(LengthBot("글자수봇"))

manager.broadcast("hello python")
manager.broadcast("AI service")
manager.report()
# 기대: 각 봇이 2건씩 처리

"✅ 이름 등록 완료"
"✅ 이름 등록 완료"
"✅ 이름 등록 완료"
[에코봇] 그대로: hello python
[대문자봇] 대문자: HELLO PYTHON
[글자수봇] 글자수: 12
[에코봇] 그대로: AI service
[대문자봇] 대문자: AI SERVICE
[글자수봇] 글자수: 10
[에코봇] 기록 개수:2
[대문자봇] 기록 개수:2
[글자수봇] 기록 개수:2


---

## 5. 📝 이해도 확인 퀴즈

**Q1.** 다형성을 "전원 버튼" 비유로 한 문장으로 설명해 보세요.
- 📌 내 답: 하나의 전원버튼이라는 기능을 통해 다양한 출력 컴퓨터 켜짐, 에어컨 켜짐, 라디오 켜짐 등 다른 리턴을 받는다.

**Q2.** `if / elif` 방식과 다형성 방식의 가장 큰 차이는 무엇인가요? (새 종류를 추가할 때를 기준으로)
- 📌 내 답: 새로운 객체를 추가 할때 if와 elif를 필요 분기 함수마다 처리 해야하지만 다형성 방식으로 진행하면 class 하나를 정의하면 된다.

**Q3.** `NotImplementedError` 와 `@abstractmethod` 의 차이는 무엇인가요? (에러가 발생하는 **시점**에 주목)
- 📌 내 답: NotImplementedError는 실행할때 나타나는 에러이고 @abstractmethod는 객체 생성시점에 발생하는 에러다

**Q4.** 덕 타이핑이란 무엇인가요? 상속이 꼭 필요한가요?
- 📌 내 답: 파이썬에서 사용할 수 있는 기능으로 상속관계가 아니여도 이름이 같은 메서드에 대해 다형성 기능을 제공하는 것

**Q5.** 아래 코드의 출력을 **먼저 예측**한 뒤 다음 셀에서 확인하세요.
- 📌 내 예측: Base->기본, X->기본, Y->기본, Z->Z (상속 없음)

In [36]:
class Base:
    def run(self):
        return "기본"

class X(Base):
    def run(self):
        return "X " + super().run()

class Y(Base):
    pass

class Z:
    def run(self):
        return "Z (상속 없음)"

for obj in [Base(), X(), Y(), Z()]:
    print(type(obj).__name__, "->", obj.run())

Base -> 기본
X -> X 기본
Y -> 기본
Z -> Z (상속 없음)


---

## 7. 📊 상속과 다형성 한눈에 정리

| | 상속  | 다형성  |
|---|---|---|
| 목적 | **중복 제거**, 공통 기능 물려주기 | **하나의 명령**으로 여러 객체 처리 |
| 핵심 문법 | `class 자식(부모):`, `super()` | 오버라이딩 + 리스트 + for 문 |
| 비유 | 본사와 지점 | 전원 버튼 |
| 관계 | 다형성의 **준비 단계** | 상속의 **활용 단계** |

### 🧩 함께 쓰는 공식 패턴

```
① 부모(추상) 클래스에 공통 메서드 이름을 정한다      ← 상속
② 자식마다 그 메서드를 다르게 구현한다              ← 오버라이딩
③ 여러 자식을 리스트에 담아 for 문으로 처리한다      ← 다형성
```

이 3단계가 **AI 서비스에서 여러 모델을 다루는 표준 설계**입니다.

---

## 8. 📁 포트폴리오 정리하기 

**GitHub 산출물**

- 파일: `week_polymorphism/shapes.py`, `week_polymorphism/ai_router.py`, `week_polymorphism/bot_manager.py`
- `README.md` 에 적을 것
  - **문제 정의**: "모델별 분기 코드(if-elif)가 늘어나 신규 모델 추가 시 기존 로직을 매번 수정해야 했다"
  - **해결**: `BaseClient` 추상 클래스 + 다형성 라우터 → **신규 모델은 클래스 추가만으로 연동**
  - **설계도**: `BaseClient` → `OpenAIClient` / `GeminiClient` / `ClaudeClient` / `LlamaClient`
  - **정량 성과**: "모델 추가 시 수정 파일 N개 → 1개(신규 클래스)로 감소, 분기문 N줄 제거"
  - **확장 계획**: Part 3 에서 이 구조에 MCP 연동과 실제 API 키(.env)를 붙인다

> 💼 면접 단골 질문: **"다형성을 실제로 어디에 써 봤나요?"**
> 👉 모범 답변 예시: **"여러 LLM 을 연동할 때 공통 인터페이스(ask)를 추상 클래스로 정의하고, 모델별 클라이언트를 자식 클래스로 구현했습니다. 덕분에 신규 모델을 추가할 때 라우터 코드를 수정하지 않았습니다."**

**오늘 배운 것**

- 다형성 = **하나의 명령, 여러 결과**
- 오버라이딩 + 리스트 + for 문 → **if 문 지옥 탈출**
- `@abstractmethod` 는 **객체 생성 시점**에 구현을 강제한다.
- 파이썬은 **덕 타이핑**이라 상속 없이도 다형성이 된다.
- `__str__` 로 `print()` 결과를 바꿀 수 있다.


> 🙌 객체지향 모두 완주했습니다! 👏
> **class → 생성자/소멸자 → 접근 제어 → 상속 → 다형성** 이 다섯 개가 AI 애플리케이션 개발의 기본 체력입니다.
